In [152]:
import pandas as pd

In [178]:
otp01 = pd.read_csv('otp_01.csv')
otp02 = pd.read_csv('otp_02.csv')
otp03 = pd.read_csv('otp_03.csv')
otp04 = pd.read_csv('otp_04.csv')
otp05 = pd.read_csv('otp_05.csv')
otp06 = pd.read_csv('otp_06.csv')
otp07 = pd.read_csv('otp_07.csv')
otp08 = pd.read_csv('otp_08.csv')
otp09 = pd.read_csv('otp_09.csv')
otp10 = pd.read_csv('otp_10.csv')
otp11 = pd.read_csv('otp_11.csv')
otp12 = pd.read_csv('otp_12.csv')

airport_codes = pd.read_csv('airport_codes.csv')
airlines = pd.read_csv('airlines.csv')
# best_airports = pd.read_csv('best_airports.csv')

In [179]:
otp_list = [otp01, otp02, otp03, otp04, otp05, otp06,
            otp07, otp08, otp09, otp10, otp11, otp12]

otp = pd.concat(otp_list, ignore_index=True)
otp

,FL_DATE,OP_CARRIER,ORIGIN,ORIGIN_STATE_ABR,DEP_DELAY,ARR_DELAY,CANCELLED,CANCELLATION_CODE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,1/1/2022 12:00:00 AM,9E,ABE,PA,NaN,NaN,1.0,B,NaN,NaN,NaN,NaN,NaN
1,1/1/2022 12:00:00 AM,9E,ABE,PA,-5.0,-25.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1/1/2022 12:00:00 AM,9E,ABE,PA,-1.0,-15.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1/1/2022 12:00:00 AM,9E,ABY,GA,-3.0,-10.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1/1/2022 12:00:00 AM,9E,ABY,GA,56.0,45.0,0.0,NaN,0.0,0.0,0.0,0.0,45.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6729120,12/31/2022 12:00:00 AM,YX,SDF,KY,108.0,99.0,0.0,NaN,20.0,0.0,0.0,0.0,79.0
6729121,12/31/2022 12:00:00 AM,YX,SRQ,FL,71.0,61.0,0.0,NaN,0.0,0.0,8.0,0.0,53.0
6729122,12/31/2022 12:00:00 AM,YX,SYR,NY,-7.0,-15.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
6729123,12/31/2022 12:00:00 AM,YX,TUL,OK,-5.0,-6.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [180]:
otp = otp.rename(str.lower, axis='columns')
otp = otp.rename(columns={"origin": "iata", "origin_state_abr": "state"})

In [181]:
airport_codes = airport_codes[airport_codes['country_code'].isin(['US'])]
airport_codes = airport_codes[airport_codes['iata'] != 'BQN']
airport_codes = airport_codes[~airport_codes['region_name'].isin(['Alaska', 'Hawaii'])].reset_index()
airport_codes = airport_codes[['region_name', 'iata', 'icao', 'airport', 'latitude', 'longitude']]
airport_codes

,region_name,iata,icao,airport,latitude,longitude
0,Alabama,AIV,KAIV,George Downer Airport,33.1065,-88.1978
1,Alabama,ALX,KALX,Thomas C. Russell Field,32.9147,-85.9630
2,Alabama,ANB,KANB,Anniston Regional Airport,33.5882,-85.8581
3,Alabama,ASN,KASN,Talladega Municipal Airport,33.5699,-86.0509
4,Alabama,AUO,KAUO,Auburn University Regional Airport,32.6151,-85.4340
...,...,...,...,...,...,...
1668,Wyoming,SAA,KSAA,Shively Field,41.4449,-106.8240
1669,Wyoming,SHR,KSHR,Sheridan County Airport,44.7692,-106.9800
1670,Wyoming,THP,KTHP,Hot Springs County-Thermopolis Municipal Airport,43.7136,-108.3900
1671,Wyoming,TOR,KTOR,Torrington Municipal Airport,42.0645,-104.1530


In [182]:
# Merge the DataFrames on ORIGIN and iata
flights = pd.merge(otp, airport_codes, left_on='iata', right_on='iata', how='left')
flights = flights.dropna(subset=['icao'])

In [183]:
flights['fl_date'] = flights['fl_date'].apply(lambda x: pd.Timestamp(x).date())

In [184]:
flights = pd.merge(flights, airlines, left_on='op_carrier', right_on='CARRIER', how='left')
flights['op_carrier'] = flights['CARRIERNAME']
flights = flights.drop(columns=['CARRIER', 'CARRIERNAME'])
flights = flights.rename(str.lower, axis='columns')
flights = flights.rename(columns={"region_name": "state_name"})
flights

,fl_date,op_carrier,iata,state,dep_delay,arr_delay,cancelled,cancellation_code,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,state_name,icao,airport,latitude,longitude
0,2022-01-01,Endeavor Air Inc.,ABE,PA,NaN,NaN,1.0,B,NaN,NaN,NaN,NaN,NaN,Pennsylvania,KABE,Lehigh Valley International Airport,40.6521,-75.4408
1,2022-01-01,Endeavor Air Inc.,ABE,PA,-5.0,-25.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Pennsylvania,KABE,Lehigh Valley International Airport,40.6521,-75.4408
2,2022-01-01,Endeavor Air Inc.,ABE,PA,-1.0,-15.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Pennsylvania,KABE,Lehigh Valley International Airport,40.6521,-75.4408
3,2022-01-01,Endeavor Air Inc.,ABY,GA,-3.0,-10.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Georgia,KABY,Southwest Georgia Regional Airport,31.5355,-84.1945
4,2022-01-01,Endeavor Air Inc.,ABY,GA,56.0,45.0,0.0,NaN,0.0,0.0,0.0,0.0,45.0,Georgia,KABY,Southwest Georgia Regional Airport,31.5355,-84.1945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6523672,2022-12-31,Republic Airline,SDF,KY,108.0,99.0,0.0,NaN,20.0,0.0,0.0,0.0,79.0,Kentucky,KSDF,Louisville International Airport (Standiford F...,38.1744,-85.7360
6523673,2022-12-31,Republic Airline,SRQ,FL,71.0,61.0,0.0,NaN,0.0,0.0,8.0,0.0,53.0,Florida,KSRQ,Sarasota-Bradenton International Airport,27.3954,-82.5544
6523674,2022-12-31,Republic Airline,SYR,NY,-7.0,-15.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,New York,KSYR,Syracuse Hancock International Airport,43.1112,-76.1063
6523675,2022-12-31,Republic Airline,TUL,OK,-5.0,-6.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Oklahoma,KTUL,Tulsa International Airport,36.1984,-95.8881


In [185]:
flights = flights[['fl_date', 'op_carrier', 'iata', 'icao', 'state', 'state_name', 'airport', 'latitude', 'longitude', 'dep_delay', 'arr_delay', 'cancelled', 'cancellation_code', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']]
flights.to_csv('flights.csv')

---

In [186]:
weather = pd.read_csv('weather_events.csv')

In [195]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8627181 entries, 0 to 8627180
Data columns (total 14 columns):
 #   Column             Dtype  
---  ------             -----  
 0   EventId            object 
 1   Type               object 
 2   Severity           object 
 3   StartTime(UTC)     object 
 4   EndTime(UTC)       object 
 5   Precipitation(in)  float64
 6   TimeZone           object 
 7   AirportCode        object 
 8   LocationLat        float64
 9   LocationLng        float64
 10  City               object 
 11  County             object 
 12  State              object 
 13  ZipCode            float64
dtypes: float64(4), object(10)
memory usage: 921.5+ MB


In [187]:
w2022 = weather[weather['StartTime(UTC)'].str.contains('2022')].copy()
w2022 = w2022.reset_index().drop(['index', 'EndTime(UTC)', 'EventId', 'Precipitation(in)', 'LocationLat', 'LocationLng', 'TimeZone', 'County', 'City', 'ZipCode'], axis=1)
w2022 = w2022.rename(str.lower, axis='columns')
w2022 = w2022.rename(columns={"starttime(utc)": "wx_date", "airportcode": "icao"})

In [188]:
# w2022[['LocationLat', 'LocationLng']] = w2022[['LocationLat', 'LocationLng']].astype(str)
# w2022['ZipCode'] = w2022['ZipCode'].apply(lambda x: x.split('.')[0])
# w2022['start_time'] = w2022['StartTime(UTC)'].apply(lambda x: pd.to_datetime(x))
# w2022['end_time'] = w2022['EndTime(UTC)'].apply(lambda x: pd.to_datetime(x))

w2022.to_csv('w2022.csv')
w2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1147630 entries, 0 to 1147629
Data columns (total 5 columns):
 #   Column    Non-Null Count    Dtype 
---  ------    --------------    ----- 
 0   type      1147630 non-null  object
 1   severity  1147630 non-null  object
 2   wx_date   1147630 non-null  object
 3   icao      1147630 non-null  object
 4   state     1147630 non-null  object
dtypes: object(5)
memory usage: 43.8+ MB


In [189]:
w2022

,type,severity,wx_date,icao,state
0,Snow,Light,2022-01-01 12:34:00,K04V,CO
1,Snow,Light,2022-01-01 16:15:00,K04V,CO
2,Storm,Severe,2022-01-06 06:54:00,K04V,CO
3,Snow,Light,2022-01-08 21:14:00,K04V,CO
4,Snow,Light,2022-01-21 21:15:00,K04V,CO
...,...,...,...,...,...
1147625,Snow,Light,2022-12-21 23:00:00,KBVR,WY
1147626,Snow,Moderate,2022-12-21 23:42:00,KBVR,WY
1147627,Cold,Severe,2022-12-21 23:53:00,KBVR,WY
1147628,Cold,Severe,2022-12-24 03:53:00,KBVR,WY


In [190]:
ap = set(flights['icao'].unique())
weather_log = w2022[w2022['icao'].isin(ap)]
# weather_log = pd.merge(weather_log, airport_codes, left_on='icao', right_on='icao', how='left')
# weather_log = weather_log.rename(columns={"region_name": "state_name"})
# weather_log = weather_log.drop(['AirportCode'], axis=1).rename(str.lower, axis='columns')
weather_log

,type,severity,wx_date,icao,state
1338,Rain,Light,2022-01-01 17:53:00,KBTR,LA
1339,Rain,Light,2022-01-02 00:53:00,KBTR,LA
1340,Rain,Light,2022-01-02 03:53:00,KBTR,LA
1341,Rain,Light,2022-01-02 06:24:00,KBTR,LA
1342,Rain,Light,2022-01-08 19:53:00,KBTR,LA
...,...,...,...,...,...
1088429,Cold,Severe,2022-11-19 06:56:00,KWYS,MT
1088430,Cold,Severe,2022-11-29 04:56:00,KWYS,MT
1088431,Cold,Severe,2022-12-16 10:56:00,KWYS,MT
1088432,Cold,Severe,2022-12-22 03:56:00,KWYS,MT


In [191]:
weather_log['wx_date'] = weather_log['wx_date'].apply(lambda x: pd.Timestamp(x).date())
weather_log = weather_log.drop_duplicates()

/var/folders/0l/s4vnzkwj7sv2h7xwjcjsbjfr0000gn/T/ipykernel_94295/3541273243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_log['wx_date'] = weather_log['wx_date'].apply(lambda x: pd.Timestamp(x).date())


In [192]:
wt_cancel = flights[flights['cancellation_code'] == 'B']
merged = pd.merge(wt_cancel, weather_log, left_on=['icao', 'fl_date'], right_on=['icao', 'wx_date'])
merged = merged.rename(columns={"state_x": "state"})
merged.columns

Index(['fl_date', 'op_carrier', 'iata', 'icao', 'state', 'state_name',
       'airport', 'latitude', 'longitude', 'dep_delay', 'arr_delay',
       'cancelled', 'cancellation_code', 'carrier_delay', 'weather_delay',
       'nas_delay', 'security_delay', 'late_aircraft_delay', 'type',
       'severity', 'wx_date', 'state_y'],
      dtype='object')

In [193]:
final_merged = merged[['wx_date', 'op_carrier', 'iata', 'icao', 'state', 'airport', 'type', 'severity']]
final_merged.to_csv('weather_log.csv')

In [194]:
merged.groupby(['airport', 'type', 'severity']).size().unstack(fill_value=0)

severity                                      Heavy  Light  Moderate  Other  \
airport                                type                                   
Aberdeen Regional Airport              Cold       0      0         0      0   
                                       Fog        0      0         1      0   
                                       Rain       0      4         0      0   
                                       Snow       2     10         4      0   
                                       Storm      0      0         0      0   
...                                             ...    ...       ...    ...   
Yeager Airport                         Snow       8     21        22      0   
Yellowstone Regional Airport           Cold       0      0         0      0   
                                       Rain       0      7         1      0   
                                       Snow       0     10         1      0   
Yuma International Airport / MCAS Yuma Rain       0      1         0      0   

severity                                      Severe  UNK  
airport                                type                
Aberdeen Regional Airport              Cold        5    0  
                                       Fog         4    0  
                                       Rain        0    0  
                                       Snow        0    0  
                                       Storm       2    0  
...                                              ...  ...  
Yeager Airport                         Snow        0    0  
Yellowstone Regional Airport           Cold        3    0  
                                       Rain        0    0  
                                       Snow        0    0  
Yuma International Airport / MCAS Yuma Rain        0    0  

[1338 rows x 6 columns]